###  Day 9: Model Optimization & Grid Search

In [7]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
# 1. Load and Scale (Same as Day 8)
data = fetch_california_housing()

# from day 8 
df = pd.DataFrame(data.data, columns=data.feature_names)
df['MedHouseVal'] = data.target
print("Dataset Overview:")
print(df.head())
print("\nBasic Statistics:")
print(df.describe())

X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# 2. Define the "Grid" of parameters to try
# We want to test different values of 'alpha'
param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0, 500.0]}
# 3. Initialize Grid Search
# cv=5 means 5-fold Cross-Validation (splitting data 5 times to be sure)
grid_search = GridSearchCV(Ridge(), param_grid, cv=5, scoring='r2')
# 4. RUN THE SEARCH
grid_search.fit(X_train_scaled, y_train)
print(f"Best Alpha Found: {grid_search.best_params_}")
print(f"Best R2 Score: {grid_search.best_score_:.4f}")

Dataset Overview:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  

Basic Statistics:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900   

In [8]:
#basic model 
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

default_model = Ridge()
default_model.fit(X_train_scaled, y_train)

X_test_scaled = scaler.transform(X_test)
y_pred_default = default_model.predict(X_test_scaled)

default_r2 = r2_score(y_test, y_pred_default)
print("Default Model R2:", default_r2)

Default Model R2: 0.5758157428913686


In [9]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test_scaled)
best_r2 = r2_score(y_test, y_pred_best)

print("Optimized Model R2:", best_r2)
best_model = grid_search.best_estimator_



Optimized Model R2: 0.5757905180002314


In [10]:
print("Default R2:", default_r2)
print("Optimized R2:", best_r2)

Default R2: 0.5758157428913686
Optimized R2: 0.5757905180002314


### Reflection
#### Why is it better to use a wider range of values (e.g., [0.1, 1, 10, 100]) first, rather than small increments (e.g., [1.1, 1.2, 1.3]) when starting a Grid Search? 
Using a wide range of values such as [0.1, 1, 10, 100] allows us to explore the overall behavior of the model across different scales of the hyperparameter. It helps identify the general region where the model performs best.

If we start with small increments like [1.1, 1.2, 1.3], we risk missing better values that lie far outside this narrow range. A wide search prevents getting stuck in a local optimum and provides a broader understanding of how the hyperparameter affects model performance.

Once the optimal range is identified, we can perform a more fine-grained search within that region for further improvement.